# All Universities — Merge & Clean

Combines the AUB raw file with the peer university merged file (output of nb 05),
then applies the same cleaning pipeline as nb 03.

**Inputs:**
- `data/raw/aub_merged_data.csv` — AUB papers (scopus + abstracts already merged)
- `data/processed/peer_unis_merged.pkl` — Lehigh, Marquette, Villanova (output of nb 05)

**Output:**
- `data/processed/all_unis_cleaned.csv`
- `data/processed/all_unis_cleaned.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Load AUB data

In [ ]:
aub_path = Path('../data/raw/aub_merged_data.csv')
aub = pd.read_csv(aub_path, low_memory=False)
aub['institution'] = 'AUB'

print(f"AUB: {aub.shape[0]:,} rows × {aub.shape[1]} cols")
print(f"Columns: {aub.columns.tolist()}")

## 2. Load peer universities data

In [ ]:
peers_path = Path('../data/processed/peer_unis_merged.pkl')
peers = pd.read_pickle(peers_path)

print(f"Peers: {peers.shape[0]:,} rows × {peers.shape[1]} cols")
print(f"Institutions: {peers['institution'].value_counts().to_dict()}")

## 3. Align columns and combine

In [ ]:
# Standardise the Citations column name if needed
for df, name in [(aub, 'AUB'), (peers, 'Peers')]:
    if 'Citations' not in df.columns and 'Cited by' in df.columns:
        df.rename(columns={'Cited by': 'Citations'}, inplace=True)
        print(f"{name}: renamed 'Cited by' -> 'Citations'")

# Identify shared columns (always keep institution)
shared_cols = [c for c in aub.columns if c in peers.columns]
if 'institution' not in shared_cols:
    shared_cols.append('institution')

print(f"\nShared columns ({len(shared_cols)}): {shared_cols}")

aub_only = set(aub.columns) - set(peers.columns)
peers_only = set(peers.columns) - set(aub.columns)
print(f"\nAUB-only columns:   {sorted(aub_only)}")
print(f"Peers-only columns: {sorted(peers_only)}")

In [ ]:
# Combine on shared columns only
combined = pd.concat(
    [aub[shared_cols], peers[shared_cols]],
    ignore_index=True
)

print(f"Combined shape: {combined.shape[0]:,} rows × {combined.shape[1]} cols")
print(f"\nRows per institution:")
print(combined['institution'].value_counts())

## 4. Remove missing critical fields

In [ ]:
print(f"Records before cleaning: {len(combined):,}")

# Check which critical columns exist
critical = ['EID', 'Abstract', 'Citations', 'Year', 'Authors']
missing_critical = [c for c in critical if c not in combined.columns]
if missing_critical:
    print(f"WARNING: missing critical columns: {missing_critical}")

present_critical = [c for c in critical if c in combined.columns]
mask = combined[present_critical].notna().all(axis=1)
df_clean = combined[mask].copy()

print(f"Records after dropping rows with missing critical fields: {len(df_clean):,}")
print(f"Removed: {len(combined) - len(df_clean):,}")

## 5. Remove duplicates

In [ ]:
print(f"Duplicate EIDs: {df_clean['EID'].duplicated().sum():,}")
df_clean = df_clean.drop_duplicates(subset='EID', keep='first')
print(f"Records after dedup: {len(df_clean):,}")

## 6. Fix data types

In [ ]:
df_clean['Year'] = pd.to_numeric(df_clean['Year'], errors='coerce').astype('Int64')
df_clean['Citations'] = pd.to_numeric(df_clean['Citations'], errors='coerce').astype('Int64')

# Drop rows where coercion produced NaN
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Year', 'Citations'])
df_clean['Year'] = df_clean['Year'].astype(int)
df_clean['Citations'] = df_clean['Citations'].astype(int)
print(f"Dropped {before - len(df_clean):,} rows where Year/Citations could not be parsed")
print(f"Year dtype: {df_clean['Year'].dtype}, Citations dtype: {df_clean['Citations'].dtype}")

## 7. Remove invalid entries

In [ ]:
before = len(df_clean)

df_clean = df_clean[
    (df_clean['Year'] >= 2000) &
    (df_clean['Year'] <= 2025) &
    (df_clean['Citations'] >= 0) &
    (df_clean['Abstract'].str.len() > 50)
]

print(f"Records after validation: {len(df_clean):,}")
print(f"Removed: {before - len(df_clean):,}")

## 8. Missing value summary

In [ ]:
key_cols = ['EID', 'Abstract', 'Citations', 'Year', 'Authors', 'Title']

print(f"{'Column':<20} {'Present':<10} {'Missing':>10} {'Missing %':>12}")
print("-" * 55)
for col in key_cols:
    if col in df_clean.columns:
        n_missing = df_clean[col].isna().sum()
        pct = n_missing / len(df_clean) * 100
        print(f"{col:<20} {'✓':<10} {n_missing:>10,} {pct:>11.1f}%")
    else:
        print(f"{col:<20} {'✗ MISSING':<10}")

## 9. Summary statistics

In [ ]:
print("=" * 50)
print("CLEANED DATASET SUMMARY")
print("=" * 50)
print(f"Total records:  {len(df_clean):,}")
print(f"Total columns:  {len(df_clean.columns)}")
print(f"Memory usage:   {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nYear range:     {df_clean['Year'].min()} – {df_clean['Year'].max()}")
print(f"Citations range: {df_clean['Citations'].min()} – {df_clean['Citations'].max()}")
print(f"Median citations: {df_clean['Citations'].median():.0f}")
print(f"Mean citations:   {df_clean['Citations'].mean():.2f}")
print(f"\nRows per institution:")
print(df_clean['institution'].value_counts())

## 10. Save

In [ ]:
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / 'all_unis_cleaned.csv'
out_pkl = out_dir / 'all_unis_cleaned.pkl'

df_clean.to_csv(out_csv, index=False)
df_clean.to_pickle(out_pkl)

print(f"Saved {len(df_clean):,} rows to:")
print(f"  {out_csv}")
print(f"  {out_pkl}")